# 00 · Download BTS On-Time (2010 → latest) and convert to Parquet
Source: https://www.transtats.bts.gov/ontime/ → *Reporting Carrier On-Time Performance (1987-present)*.

For each month: download zip → extract CSV → keep selected columns as Parquet → delete CSV.
Safe to re-run: months already converted are skipped, failed months are listed at the end.

In [2]:
import os, time, zipfile, shutil, urllib.request, urllib.error
from pathlib import Path
import duckdb, pandas as pd

In [3]:
# ---- settings -------------------------------------------------------------
START, END = "2010-01", "2026-08"      # inclusive, YYYY-MM
TEST_ONE_MONTH = False                  # True: only process START (dry run). Set False for the full run.
KEEP_ZIPS = False                      # True keeps zips (needs more disk)
PAUSE_SECONDS = 2                      # politeness delay between downloads
URL_TEMPLATE = ("https://transtats.bts.gov/PREZIP/"
                "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{y}_{m}.zip")
DATA = Path(os.path.expanduser("~/Desktop/ADSP32030_Scalable/Project/data/flights"))

In [4]:
RAW, PQ = DATA/"raw", DATA/"parquet"
RAW.mkdir(parents=True, exist_ok=True); PQ.mkdir(parents=True, exist_ok=True)
LOG = DATA/"conversion_log.csv"

KEEP = ["Year","Month","DayofMonth","DayOfWeek","FlightDate","Reporting_Airline",
        "IATA_CODE_Reporting_Airline","Tail_Number","Flight_Number_Reporting_Airline",
        "Origin","OriginState","Dest","DestState","CRSDepTime","DepTime","DepDelay","DepDel15",
        "TaxiOut","WheelsOff","WheelsOn","TaxiIn","CRSArrTime","ArrTime","ArrDelay","ArrDel15",
        "Cancelled","CancellationCode","Diverted","CRSElapsedTime","ActualElapsedTime","AirTime",
        "Distance","CarrierDelay","WeatherDelay","NASDelay","SecurityDelay","LateAircraftDelay"]
NUMERIC = {"Year","Month","DayofMonth","DayOfWeek","CRSDepTime","DepTime","DepDelay","DepDel15","TaxiOut",
           "WheelsOff","WheelsOn","TaxiIn","CRSArrTime","ArrTime","ArrDelay","ArrDel15","Cancelled","Diverted",
           "CRSElapsedTime","ActualElapsedTime","AirTime","Distance","CarrierDelay","WeatherDelay","NASDelay",
           "SecurityDelay","LateAircraftDelay"}

def month_range(start, end):
    y, m = map(int, start.split("-")); ye, me = map(int, end.split("-"))
    while (y, m) <= (ye, me):
        yield y, m
        m += 1
        if m == 13: y, m = y + 1, 1

def download(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "adsp32030-flight-project/1.0"})
    with urllib.request.urlopen(req, timeout=180) as r, open(dest, "wb") as f:
        while chunk := r.read(1 << 20):      # 1 MB at a time: never holds the file in memory
            f.write(chunk)

def convert(csv_path, out):
    con = duckdb.connect()
    src = f"read_csv('{csv_path}', header=true, sample_size=-1, ignore_errors=true)"
    have = {r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {src}").fetchall()}
    sel = ", ".join((f'TRY_CAST("{c}" AS DOUBLE) AS "{c}"' if c in NUMERIC else f'"{c}"') for c in KEEP if c in have)
    con.execute(f"COPY (SELECT {sel} FROM {src}) TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    n = con.execute(f"SELECT count(*) FROM read_parquet('{out}')").fetchone()[0]
    con.close()
    return n

def log_row(row):
    df = pd.DataFrame([row])
    if LOG.exists():
        old = pd.read_csv(LOG); old = old[old.file != row["file"]]
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(LOG, index=False)

## Run

In [7]:
months = list(month_range(START, END))
if TEST_ONE_MONTH: months = months[:1]
failed = []
for y, m in months:
    tag = f"{y}-{m:02d}"
    out = PQ/f"bts_{tag}.parquet"
    if out.exists():
        print(f"[skip] {tag}"); continue
    zpath = RAW/f"{tag}.zip"
    try:
        if not zpath.exists():
            print(f"[get ] {tag}"); download(URL_TEMPLATE.format(y=y, m=m), zpath)
        with zipfile.ZipFile(zpath) as zf:
            name = next(n for n in zf.namelist() if n.lower().endswith(".csv"))
            zf.extract(name, RAW)
        csv = RAW/name
        n = convert(csv, out)
        row = dict(file=f"bts_{tag}", rows=n, csv_mb=csv.stat().st_size/1e6, parquet_mb=out.stat().st_size/1e6)
        log_row(row); csv.unlink()
        if not KEEP_ZIPS: zpath.unlink()
        print(f"[ok  ] {tag}: {n:,} rows | CSV {row['csv_mb']:.0f} MB -> Parquet {row['parquet_mb']:.0f} MB")
    except (urllib.error.URLError, zipfile.BadZipFile, StopIteration, duckdb.Error, OSError) as e:
        failed.append(tag); print(f"[FAIL] {tag}: {e}")
        if zpath.exists() and not KEEP_ZIPS: zpath.unlink()
    time.sleep(PAUSE_SECONDS)
print("failed months:", failed or "none")

[skip] 2010-01
[skip] 2010-02
[skip] 2010-03
[skip] 2010-04
[skip] 2010-05
[skip] 2010-06
[skip] 2010-07
[skip] 2010-08
[skip] 2010-09
[skip] 2010-10
[skip] 2010-11
[skip] 2010-12
[skip] 2011-01
[skip] 2011-02
[skip] 2011-03
[skip] 2011-04
[skip] 2011-05
[skip] 2011-06
[skip] 2011-07
[skip] 2011-08
[skip] 2011-09
[skip] 2011-10
[skip] 2011-11
[skip] 2011-12
[skip] 2012-01
[skip] 2012-02
[skip] 2012-03
[skip] 2012-04
[skip] 2012-05
[skip] 2012-06
[skip] 2012-07
[skip] 2012-08
[skip] 2012-09
[skip] 2012-10
[skip] 2012-11
[skip] 2012-12
[skip] 2013-01
[skip] 2013-02
[skip] 2013-03
[skip] 2013-04
[skip] 2013-05
[skip] 2013-06
[skip] 2013-07
[skip] 2013-08
[skip] 2013-09
[skip] 2013-10
[skip] 2013-11
[skip] 2013-12
[skip] 2014-01
[skip] 2014-02
[skip] 2014-03
[skip] 2014-04
[skip] 2014-05
[skip] 2014-06
[skip] 2014-07
[skip] 2014-08
[skip] 2014-09
[skip] 2014-10
[skip] 2014-11
[skip] 2014-12
[skip] 2015-01
[skip] 2015-02
[skip] 2015-03
[skip] 2015-04
[skip] 2015-05
[skip] 2015-06
[skip] 201

## Does it meet the big-data rule?

In [8]:
log = pd.read_csv(LOG)
total_rows = log.rows.sum(); raw_gb = log.csv_mb.sum()/1000; pq_gb = log.parquet_mb.sum()/1000
free_gb = shutil.disk_usage(DATA).free/1e9
try:
    import psutil; ram_gb = psutil.virtual_memory().total/1e9
except ImportError:
    ram_gb = os.sysconf("SC_PAGE_SIZE")*os.sysconf("SC_PHYS_PAGES")/1e9
print(f"months loaded: {len(log)}   rows: {total_rows/1e6:,.1f}M")
print(f"raw CSV (all columns): {raw_gb:,.1f} GB   Parquet (trimmed): {pq_gb:,.1f} GB   ratio x{raw_gb/pq_gb:.1f}")
print(f"your RAM: {ram_gb:,.0f} GB   free disk: {free_gb:,.0f} GB")
print("raw CSV bigger than RAM:", raw_gb > ram_gb, "| raw CSV bigger than free disk:", raw_gb > free_gb)
if len(log) < 190: print("(partial load: scale up before quoting totals)")
log.tail()

months loaded: 200   rows: 105.6M
raw CSV (all columns): 47.6 GB   Parquet (trimmed): 2.0 GB   ratio x24.2
your RAM: 17 GB   free disk: 58 GB
raw CSV bigger than RAM: True | raw CSV bigger than free disk: False


,file,rows,csv_mb,parquet_mb
195,bts_2026-05,611735,276.871221,11.638758
196,bts_2026-06,607577,275.276886,11.514040
197,bts_2010-04,529330,237.957963,9.098229
198,bts_2026-08,608389,275.779587,11.788746
199,bts_2026-07,631970,286.311394,11.720673


## For the proposal / slides
- Rows (millions) and months loaded, raw CSV GB vs RAM and disk (output above)
- Parquet size and the compression ratio
- Failed months and why

In [10]:
con = duckdb.connect()
glob = str(PQ / "*.parquet")
top = con.execute(f"""
    SELECT Origin AS airport,
           count(*) AS flights,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct_of_all,
           round(100.0 * sum(count(*)) OVER (ORDER BY count(*) DESC) / sum(count(*)) OVER (), 1) AS cumulative_pct
    FROM read_parquet('{glob}')
    GROUP BY Origin
    ORDER BY flights DESC
    LIMIT 50
""").df()
top.index = top.index + 1
n_airports = con.execute(f"SELECT count(DISTINCT Origin) FROM read_parquet('{glob}')").fetchone()[0]
print(f"distinct origin airports: {n_airports}")
top

distinct origin airports: 410


,airport,flights,pct_of_all,cumulative_pct
1,ATL,5971746,5.65,5.7
2,ORD,4831539,4.57,10.2
3,DFW,4493443,4.25,14.5
4,DEN,4178538,3.96,18.4
5,LAX,3356895,3.18,21.6
6,PHX,2861016,2.71,24.3
7,CLT,2745871,2.60,26.9
8,LAS,2589039,2.45,29.4
9,SFO,2481112,2.35,31.7
10,IAH,2452004,2.32,34.0
